In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "spark"))

In [15]:
#import relevant packages
import sys
import os
import argparse 
from pathlib import Path
from pyspark.sql import SparkSession    
from pyspark.sql.functions import col

from spark.framework.iceberg.schema_loader import load_platform_schema
#from spark.framework.iceberg.iceberg_utils import (ensure_namespace_exists)
from spark.framework.iceberg.table_manager import create_table_if_not_exists, ensure_namespace_exists, write_to_iceberg
#fetch the project root directory 
#PROJECT_ROOT = Path(__file__).resolve().parents[1]
#sys.path.append(str(PROJECT_ROOT))
from spark.common.config_loader import load_config
from spark.framework.metadata.entity_config_loader import (load_entity_config, )
from spark.framework.transformation.silver_builder import (build_silver)
from spark.framework.metadata.schema_generator import generate_schema

from pyspark.sql.types import (
    StructType,
    StructField,
    BinaryType,
    StringType,
    IntegerType,
    LongType,
    TimestampType
)
from spark.framework.spark.spark_session import create_spark_session

BRONZE_SCHEMA = StructType([
    StructField("key", BinaryType(), True),
    StructField("value", BinaryType(), True),
    StructField("topic", StringType(), False),
    StructField("partition", IntegerType(), False),
    StructField("offset", LongType(), False),
    StructField("timestamp", TimestampType(), True),
    StructField("timestampType", StringType(), True)
])

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    BinaryType,
    StringType,
    IntegerType,
    LongType,
    TimestampType
)
from spark.framework.spark.spark_session import create_spark_session
from spark.framework.logging.logger import get_logger
from pyspark.sql.functions import col, size
from spark.framework.validation.splitter import split_valid_invalid
from spark.framework.pipeline.pipeline_context import PipelineContext
from spark.framework.persistence.persist_silver import persist_silver
from spark.framework.dlq.dlq_schema import DLQ_SCHEMA

logger = get_logger(
            "silver_Stream",
            env_config
            )

In [5]:
entity_name = "customer"

    ##Read environment variable for ENV, default to "local" if not set
env = os.getenv("ENV", "local")
print(f"Environment: {env}")

print(f"Starting Silver Stream for Entity: {entity_name}")

    #Load configuration for environment and entity
env_config = load_config(env)
entity_config = load_entity_config(entity_name)

    #silver schema path
silver_schema_path = PROJECT_ROOT / "configs" / "entities" / f"{entity_name}.yaml"

    #create the bronze table and silver table names using the entity name from the entity configuration
catalog_name = env_config['iceberg']['catalog_name']
bronze_table = f"{catalog_name}.bronze.{entity_config['entity_name']}"
print(f"Bronze Table: {bronze_table}")
silver_table = f"{catalog_name}.silver.{entity_config['entity_name']}"
print(f"Silver Table: {silver_table}")
    
    #create a checkpoint path for the silver stream using the entity name from the entity configuration
checkpoint_path = (
    f"{env_config['storage']['checkpoint_root_path']}/silver/{entity_config['entity_name']}"
    )
print(f"Checkpoint Path: {checkpoint_path}") 

Environment: local
Starting Silver Stream for Entity: customer
Bronze Table: insightflow.bronze.customer
Silver Table: insightflow.silver.customer
Checkpoint Path: gs://insightflowai-data-prod/checkpoints/silver/customer


In [6]:
 #create Spark Session
spark = create_spark_session("Silver Stream", env)
        
print("1. Spark Session Created")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/19 21:23:26 WARN Utils: Your hostname, Sauravs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.7 instead (on interface en0)
26/08/19 21:23:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/sauravpandey/Projects/streaming/subscription-platform/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/sauravpandey/.ivy2.5.2/cache
The jars for the packages stored in: /Users/sauravpandey/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
com.google.cloud.bigdataoss#gcs-connector added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9c344fd8-3e56-4216-9b09-48cd971d1fdf;1.0
	confs: [default]
	foun

1. Spark Session Created


In [7]:
spark.sql("DROP TABLE insightflow.silver.customer")

DataFrame[]

In [7]:
#Crete bronze schema to allow spark to read the bronze parquet files and convert to string format for silver layer processing
silver_schema = load_platform_schema(silver_schema_path)  

silver_schema

{'schema_version': 1,
 'entity_name': 'customer',
 'source': {'topic': 'insightflow.public.customer'},
 'business_keys': 'customer_id',
 'columns': {'customer_id': {'datatype': 'string',
   'nullable': False,
   'business_key': True},
  'customer_name': {'datatype': 'string', 'nullable': False},
  'customer_status': {'datatype': 'string',
   'nullable': False,
   'allowed_values': ['ACTIVE', 'INACTIVE']},
  'industry': {'datatype': 'string', 'nullable': True},
  'customer_size': {'datatype': 'string', 'nullable': True},
  'customer_tier': {'datatype': 'string',
   'nullable': True,
   'allowed_values': ['Gold',
    'Platinum',
    'Silver',
    'Diamond',
    'GOLD',
    'LOCAL TESTING']},
  'customer_start_date': {'datatype': 'date',
   'nullable': True,
   'source_format': 'epoch_days'},
  'customer_end_date': {'datatype': 'date',
   'nullable': True,
   'source_format': 'epoch_days'},
  'created_timestamp': {'datatype': 'timestamp',
   'nullable': False,
   'source_format': 'epoch_m

In [9]:
print("entity_config: ", entity_config)
print("silver_schema: ", silver_schema)

entity_config:  {'schema_version': 1, 'entity_name': 'customer', 'source': {'topic': 'insightflow.public.customer'}, 'business_keys': 'customer_id', 'columns': {'customer_id': {'datatype': 'string', 'nullable': False, 'business_key': True}, 'customer_name': {'datatype': 'string', 'nullable': False}, 'customer_status': {'datatype': 'string', 'nullable': False, 'allowed_values': ['ACTIVE', 'INACTIVE']}, 'industry': {'datatype': 'string', 'nullable': True}, 'customer_size': {'datatype': 'string', 'nullable': True}, 'customer_tier': {'datatype': 'string', 'nullable': True, 'allowed_values': ['Gold', 'Platinum', 'Silver']}, 'customer_start_date': {'datatype': 'date', 'nullable': True, 'source_format': 'epoch_days'}, 'customer_end_date': {'datatype': 'date', 'nullable': True, 'source_format': 'epoch_days'}, 'created_timestamp': {'datatype': 'timestamp', 'nullable': False, 'source_format': 'epoch_micros'}, 'updated_timestamp': {'datatype': 'timestamp', 'nullable': False, 'source_format': 'epo

In [8]:
entity_schema = generate_schema(entity_config)
print("Entity Schema: ", entity_schema)

Entity Schema:  StructType([StructField('customer_id', StringType(), False), StructField('customer_name', StringType(), False), StructField('customer_status', StringType(), False), StructField('industry', StringType(), True), StructField('customer_size', StringType(), True), StructField('customer_tier', StringType(), True), StructField('customer_start_date', IntegerType(), True), StructField('customer_end_date', IntegerType(), True), StructField('created_timestamp', LongType(), False), StructField('updated_timestamp', LongType(), False)])


In [9]:
entity_schema = generate_schema(entity_config)
print("Entity Schema: ", entity_schema)

Entity Schema:  StructType([StructField('customer_id', StringType(), False), StructField('customer_name', StringType(), False), StructField('customer_status', StringType(), False), StructField('industry', StringType(), True), StructField('customer_size', StringType(), True), StructField('customer_tier', StringType(), True), StructField('customer_start_date', IntegerType(), True), StructField('customer_end_date', IntegerType(), True), StructField('created_timestamp', LongType(), False), StructField('updated_timestamp', LongType(), False)])


In [10]:
    #ensure the silver namespace exists in the iceberg catalog, if not create it
ensure_namespace_exists(spark, catalog_name, "silver")



In [13]:
    #create the silver table in the iceberg catalog if it does not exist, using the silver schema and the silver path
create_table_if_not_exists(
    spark=spark,
    catalog=catalog_name,
    namespace="silver",
    table_name=entity_name,
    schema=silver_schema,
    logger=None
    )

AttributeError: 'NoneType' object has no attribute 'info'

In [14]:
 #Read Bronze data from the bronze storage path in parquet format and convert to string format for Silver layer processing
bronze_df = (
        spark.read
        .table(bronze_table)
    )
print("2. Bronze Stream Created\n",bronze_df.show(10, truncate=False))

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
bronze_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = false)
 |-- topic: string (nullable = false)
 |-- partition: integer (nullable = false)
 |-- offset: long (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = false)



In [12]:
 #Read Bronze data from the bronze storage path in parquet format and convert to string format for Silver layer processing
bronze_df = (
    spark.readStream
    .table(bronze_table)
)
print("2. Bronze Stream Created\n",bronze_df)

2. Bronze Stream Created
 DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]


In [7]:
from framework.logging.logger import get_logger
#configure logging
logger = get_logger(
    "silver_Stream",
    env_config
    )

In [8]:
#call silver builder function to process the bronze data and and transform for silver layer processing and write to silver storage path in parquet format
valid_df, invalid_df = build_silver(bronze_df, silver_schema, env_config, entity_name, logger=logger)
print("Silver DF Built")

2026-07-09 16:55:39,636 | INFO     | silver_Stream | Starting Silver Builder...
2026-07-09 16:55:39,639 | INFO     | silver_Stream | Bronze Data Read Successfully from stream file
2026-07-09 16:55:39,640 | INFO     | silver_Stream | Silver Data will be written to: gs://insightflowai-data-prod/silver/customer
2026-07-09 16:55:39,640 | INFO     | silver_Stream | Extracting Required Fields for Silver Layer Processing...
2026-07-09 16:55:39,684 | INFO     | silver_Stream | calling parse_debezium function to extract before, after, op, source, ts_ms from raw_payload
Running Debezium Parser
2026-07-09 16:55:39,706 | INFO     | silver_Stream | Debezium Parser Completed
calling map_entity function to convert cdc event into entity-specific silver records
2026-07-09 16:55:39,707 | INFO     | silver_Stream | Running Entity Mapper
Running Normalizer
2026-07-09 16:55:39,789 | INFO     | silver_Stream | Normalizer Completed
2026-07-09 16:55:39,789 | INFO     | silver_Stream | Entity Mapping Completed

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+-------------------------+-----------------------+-----------------------+-----------------------+-------------+--------------------+
|customer_id|customer_name|customer_status|     industry|customer_size|customer_tier|customer_start_date|customer_end_date|   created_timestamp|   updated_timestamp| op|              ts_ms|               topic|partition|offset|           timestamp|__raw_customer_start_date|__raw_customer_end_date|__raw_created_timestamp|__raw_updated_timestamp|  __raw_ts_ms|    Validation_Error|
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+---------

In [15]:
valid_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- customer_size: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- customer_start_date: date (nullable = true)
 |-- customer_end_date: date (nullable = true)
 |-- created_timestamp: timestamp (nullable = true)
 |-- updated_timestamp: timestamp (nullable = true)
 |-- op: string (nullable = true)
 |-- ts_ms: timestamp (nullable = true)
 |-- topic: string (nullable = false)
 |-- partition: integer (nullable = false)
 |-- offset: long (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- Validation_Error: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- rule: string (nullable = false)
 |    |    |-- column: string (nullable = false)
 |    |    |-- error_message: string (nullable = false)



In [16]:
invalid_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- customer_size: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- customer_start_date: date (nullable = true)
 |-- customer_end_date: date (nullable = true)
 |-- created_timestamp: timestamp (nullable = true)
 |-- updated_timestamp: timestamp (nullable = true)
 |-- op: string (nullable = true)
 |-- ts_ms: timestamp (nullable = true)
 |-- topic: string (nullable = false)
 |-- partition: integer (nullable = false)
 |-- offset: long (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- Validation_Error: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- rule: string (nullable = false)
 |    |    |-- column: string (nullable = false)
 |    |    |-- error_message: string (nullable = false)



In [9]:
valid_df.show(10, truncate=False)

26/07/09 16:55:57 WARN DAGScheduler: Broadcasting large task binary with size 1708.4 KiB
26/07/09 16:55:58 WARN DAGScheduler: Broadcasting large task binary with size 1708.4 KiB


+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+---------------------------+---------+------+-----------------------+----------------+
|customer_id|customer_name|customer_status|industry     |customer_size|customer_tier|customer_start_date|customer_end_date|created_timestamp         |updated_timestamp         |op |ts_ms              |topic                      |partition|offset|timestamp              |Validation_Error|
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+---------------------------+---------+------+-----------------------+----------------+
|CUST1003   |Netflix      |ACTIVE         |Entertainment|Large        |Silver       |2026-07-03         |NULL             |2026-07-03 12

In [10]:
invalid_df.show(10, truncate=False)

+-----------+-------------+---------------+----------+-------------+-------------+-------------------+-----------------+-------------------------+--------------------------+---+-------------------+---------------------------+---------+------+-----------------------+--------------------------------------------------------------------------------------------------------+
|customer_id|customer_name|customer_status|industry  |customer_size|customer_tier|customer_start_date|customer_end_date|created_timestamp        |updated_timestamp         |op |ts_ms              |topic                      |partition|offset|timestamp              |Validation_Error                                                                                        |
+-----------+-------------+---------------+----------+-------------+-------------+-------------------+-----------------+-------------------------+--------------------------+---+-------------------+---------------------------+---------+------+--------------

In [18]:
silver_df = silver_df.drop(
    "topic",
    "partition",
    "offset",
    "timestamp"
)

In [16]:
silver_df

DataFrame[customer_id: string, customer_name: string, customer_status: string, industry: string, customer_size: string, customer_tier: string, customer_start_date: date, customer_end_date: date, created_timestamp: timestamp, updated_timestamp: timestamp, op: string, ts_ms: timestamp]

In [19]:
 #write to silver storage path in parquet format with metadata and payload fields extracted from the bronze layer parquet data
query = (
    silver_df.writeStream
      .foreachBatch(
            lambda batch_df, batch_id:
                write_to_iceberg(
                    batch_df=batch_df,
                    batch_id=batch_id,
                    table_name=silver_table,
                    mode="append"
                    
                )
      )
      .option("mergeSchema", "true")
      .option(
          "checkpointLocation",
          checkpoint_path
      )
      .start()
)

26/07/05 00:34:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [24]:
query.isActive

False

In [16]:
spark.sql("select * from insightflow.silver.customer").show(10, truncate=False)

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+
|customer_id|customer_name|customer_status|industry     |customer_size|customer_tier|customer_start_date|customer_end_date|created_timestamp         |updated_timestamp         |op |ts_ms              |
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+
|CUST1001   |OpenAI       |ACTIVE         |Technology   |Enterprise   |Platinum     |2026-07-03         |NULL             |2026-07-03 10:46:16.108056|2026-07-03 10:46:16.108056|c  |2026-07-03 10:46:16|
|CUST1002   |Microsoft    |ACTIVE         |Technology   |Enterprise   |Gold         |2026-07-03         |NULL             |2026-07-03 11:57:05.05155 |2026-07-03 11:57:05.05155 |c  |2026-07-03 

In [12]:
bronze = spark.read.table("insightflow.bronze.customer")


In [13]:
bronze.selectExpr(
    "CAST(value AS STRING) AS raw_payload"
).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
spark.read.table("insightflow.bronze.customer") \
    .selectExpr("CAST(value AS STRING) AS raw_payload") \
    .show(1, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:

valid_df.show(10, truncate=False)

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+---------------------------+---------+------+-----------------------+-------------------------+-----------------------+-----------------------+-----------------------+-------------+----------------+
|customer_id|customer_name|customer_status|industry     |customer_size|customer_tier|customer_start_date|customer_end_date|created_timestamp         |updated_timestamp         |op |ts_ms              |topic                      |partition|offset|timestamp              |__raw_customer_start_date|__raw_customer_end_date|__raw_created_timestamp|__raw_updated_timestamp|__raw_ts_ms  |Validation_Error|
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------

In [15]:
valid_df

DataFrame[customer_id: string, customer_name: string, customer_status: string, industry: string, customer_size: string, customer_tier: string, customer_start_date: date, customer_end_date: date, created_timestamp: timestamp, updated_timestamp: timestamp, op: string, ts_ms: timestamp, topic: string, partition: int, offset: bigint, timestamp: timestamp, __raw_customer_start_date: int, __raw_customer_end_date: int, __raw_created_timestamp: bigint, __raw_updated_timestamp: bigint, __raw_ts_ms: string, Validation_Error: array<struct<rule:string,column:string,error_message:string>>]

In [13]:
for column, metadata in entity_config.get("columns", {}).items():
        print(metadata.get('allowed_values'))
        allowed_values = metadata.get('allowed_values')
        print(allowed_values)

None
None
None
None
['ACTIVE']
['ACTIVE']
None
None
None
None
['Gold', 'Platinum', 'Silver']
['Gold', 'Platinum', 'Silver']
None
None
None
None
None
None
None
None


In [14]:
from pyspark.sql.functions import col
from framework.validation.constants import TEMP_COLUMN_PREFIX
from framework.validation.utils import append_validation_error

RULE_NAME = "allowed_values"
for column, metadata in entity_config.get("columns", {}).items():
        allowed_values = metadata.get('allowed_values')
        print(allowed_values)
        if "allowed_values" not in metadata:
            continue
        
        raw_column =f'{TEMP_COLUMN_PREFIX}{column}'
        print(raw_column)
        valid_df_chk = append_validation_error(
            df = valid_df,
            condition = (col(column).isNotNull() & (~col(column).isin(allowed_values))),
            rule=RULE_NAME,
            column=column,
            error_message = f"{column} contains invalid values. Allowed values: {allowed_values}"
        )


None
None
['ACTIVE']
__raw_customer_status
None
None
['Gold', 'Platinum', 'Silver']
__raw_customer_tier
None
None
None
None


In [15]:
valid_df_chk.show()

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+----------------+
|customer_id|customer_name|customer_status|     industry|customer_size|customer_tier|customer_start_date|customer_end_date|   created_timestamp|   updated_timestamp| op|              ts_ms|               topic|partition|offset|           timestamp|Validation_Error|
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+----------------+
|   CUST1001|       OpenAI|         ACTIVE|   Technology|   Enterprise|     Platinum|         2026-07-03|             NULL|2026-07-03 10:46:...|2026-07-03 10:46:...|  c|2026-07-03 10:46:16|insightflow.p

In [12]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, explode


from framework.logging.logger import get_logger
#configure logging
logger = get_logger(
    "silver_Stream",
    env_config
    )

def summarize(
              valid_df: DataFrame
              , invalid_df): 
    """ LOG VALIDATION SUMMARY FOR EVERY MICRO-BATCH """
    valid_count = valid_df.count()
    invalid_count = invalid_df.count()
    #total_count_raw = silver_df.count()
    total_count = valid_count + invalid_count

    logger.info("=" * 60)
    logger.info("validation summary")
    logger.info("=" * 60)
    logger.info(f"Total Valid Records : {valid_count}")
    logger.info(f"Total Inalid Records : {invalid_count}")

    if invalid_count == 0:
        logger.info("No Validation failure detected.")
        return
    rule_summary = (
        invalid_df
        .select(explode(col("Validation_Error")).alias("error"))
        .groupBy("error.rule")
        .count()
        .collect()
    )
    for row in rule_summary:

        logger.info(
            f"{row['rule']} : {row['count']}"
        )

        logger.info("=" * 60)
    


     


In [ ]:
summarize(valid_df, invalid_df)

26/07/09 16:56:47 WARN DAGScheduler: Broadcasting large task binary with size 1676.4 KiB


2026-07-09 16:56:50,201 | INFO     | silver_Stream | ============================================================
2026-07-09 16:56:50,202 | INFO     | silver_Stream | validation summary
2026-07-09 16:56:50,203 | INFO     | silver_Stream | ============================================================
2026-07-09 16:56:50,203 | INFO     | silver_Stream | Total Valid Records : 6
2026-07-09 16:56:50,203 | INFO     | silver_Stream | Total Inalid Records : 1


26/07/09 16:56:51 WARN DAGScheduler: Broadcasting large task binary with size 1859.5 KiB


2026-07-09 16:56:52,692 | INFO     | silver_Stream | allowed_values : 1
2026-07-09 16:56:52,692 | INFO     | silver_Stream | ============================================================


26/07/09 16:56:52 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB


26/07/09 18:41:17 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 256459 ms exceeds timeout 120000 ms
26/07/09 18:41:17 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/09 18:41:23 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at o

In [63]:
from pyspark.sql import DataFrame
from framework.validation.constants import TEMP_COLUMN_PREFIX
from framework.validation.utils import append_validation_error
from pyspark.sql.functions import col

RULE_NAME = "allowed_values"

def validate_allowed_values(
        df: DataFrame,
        entity_config: dict
)-> DataFrame:
    print("Executing Allowed values validations")
    """
    Validate the DataFrame for allowed values constraints based on the entity configuration.

    Args:
        df (DataFrame): The input DataFrame to validate.
        entity_config (dict): The configuration dictionary containing validation rules.

    Returns:
        DataFrame: The validated DataFrame with allowed values constraints applied.
    """
    # Implementation of the allowed values validation logic goes here
    for column, metadata in entity_config.get("columns", {}).items():
        allowed_values = metadata.get('allowed_values')
        print(allowed_values)
        if "allowed_values" not in metadata:
            continue
        
        
        valid_df = append_validation_error(
            df = df,
            condition = (col(column).isNotNull() & (~col(column).isin(allowed_values))),
            rule=RULE_NAME,
            column=column,
            error_message = f"{column} contains invalid values. Allowed values: {allowed_values}"
        )

    return valid_df   

In [64]:
df = validate_allowed_values(valid_df, entity_config)

Executing Allowed values validations
None
None
['ACTIVE']
None
None
['Gold', 'Platinum', 'Silver']
None
None
None
None


In [66]:
df.show()

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+----------------+
|customer_id|customer_name|customer_status|     industry|customer_size|customer_tier|customer_start_date|customer_end_date|   created_timestamp|   updated_timestamp| op|              ts_ms|               topic|partition|offset|           timestamp|Validation_Error|
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+----------------+
|   CUST1002|    Microsoft|         ACTIVE|   Technology|   Enterprise|         Gold|         2026-07-03|             NULL|2026-07-03 11:57:...|2026-07-03 11:57:...|  c|2026-07-03 11:57:05|insightflow.p

In [50]:
print(f"{allowed_values}")

None


In [51]:

from pyspark.sql.functions import array, struct, when, array, array_union, lit
RULE_NAME = "allowed_values"
column = "customer_status"
error_message = f"{column} contains invalid values. Allowed values: ACTIVE"
rule = RULE_NAME
condition = (col(column).isNotNull() & (~col(column).isin('ACTIVE')))
error = array(
        struct(
            lit(rule).alias("rule"),
            lit(column).alias("column"),
            lit(error_message).alias("error_message")
        )
    )

In [59]:
condition

Column<'and(isNotNull(customer_status), !(in(customer_status, 'ACTIVE')))'>

In [53]:
from pyspark.sql.functions import when
df_test = df.withColumn(
        "Validation_Error",
        when(condition, when(col("Validation_Error").isNull(), error).otherwise(
                        array_union(col("Validation_Error"), error))).otherwise(
                            col("Validation_Error"))
                            )

In [54]:
df_test.show()

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+--------------------+
|customer_id|customer_name|customer_status|     industry|customer_size|customer_tier|customer_start_date|customer_end_date|   created_timestamp|   updated_timestamp| op|              ts_ms|               topic|partition|offset|           timestamp|    Validation_Error|
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------+--------------------+---+-------------------+--------------------+---------+------+--------------------+--------------------+
|   CUST1002|    Microsoft|         ACTIVE|   Technology|   Enterprise|         Gold|         2026-07-03|             NULL|2026-07-03 11:57:...|2026-07-03 11:57:...|  c|2026-07-03 11:57:05|i

In [31]:
spark.read.table("insightflow.dlq.records") \
    .show(100, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [28]:
spark.read.table("insightflow.silver.customer") \
    .show(100, truncate=False)

+-----------+------------------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+
|customer_id|customer_name           |customer_status|industry     |customer_size|customer_tier|customer_start_date|customer_end_date|created_timestamp         |updated_timestamp         |op |ts_ms              |
+-----------+------------------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+
|CUST1001   |OpenAI                  |ACTIVE         |Technology   |Enterprise   |Platinum     |2026-07-03         |NULL             |2026-07-03 10:46:16.108056|2026-07-03 10:46:16.108056|r  |2026-07-14 09:37:25|
|CUST1003   |Netflix                 |ACTIVE         |Entertainment|Large        |Silver       |2026-07-03         |NULL             |2026-07-03 12:

In [32]:
df = spark.read.table("insightflow.dlq.records")

In [ ]:
df.rdd

MapPartitionsRDD[57] at javaToPython at NativeMethodAccessorImpl.java:0

26/07/25 11:36:01 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 768642 ms exceeds timeout 120000 ms
26/07/25 11:36:01 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/25 11:36:10 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at o